# Day 32 — 첫 baseline 모델: 로지스틱 회귀

설명은 학습 사이트(또는 원본 레슨)에서 읽고, 코드는 이 노트북에서 위에서 아래로 차례대로 실행하세요. 앞 셀이 만든 변수와 파일을 뒤 셀이 씁니다.

원본 레슨: [lessons/week-5/day-32.md](https://github.com/evansuhohyun-927/neurodegeneration-bioinformatics/blob/main/lessons/week-5/day-32.md)

In [ ]:
# 레슨 코드가 쓰는 폴더를 미리 만든다 (Colab에서는 /content 아래에 생긴다)
import os
for d in ["data", "figures", "reports", "results", "../data", "../figures", "../reports"]:
    os.makedirs(d, exist_ok=True)

## 예시로 이해하기

### 예시 1 — 재료 준비: X, y, groups와 참가자 단위 나누기

In [ ]:
import numpy as np
import pandas as pd

FEATURES = ["stride_time_cv", "accel_rms", "tremor_band_power",
            "jerk_mean", "step_regularity", "arm_swing_asym"]

def make_demo_table(seed=42, n_participants=50, n_repeats=6):
    """연습용 가상 feature table. 실제 환자 데이터가 아니다."""
    rng = np.random.default_rng(seed)
    pd_shift = np.array([0.6, -0.5, 0.6, 0.2, -0.4, 0.3])   # PD군 평균이 이동하는 정도
    rows = []
    for i in range(n_participants):
        group = "PD" if i < n_participants // 2 else "HC"
        age = rng.normal(68 if group == "PD" else 62, 7)
        sex = rng.choice(["F", "M"])
        person = rng.normal(0, 1.0, size=len(FEATURES))      # 그 사람 고유의 '움직임 지문'
        for rep in range(1, n_repeats + 1):
            values = person + rng.normal(0, 0.25, size=len(FEATURES))  # 측정마다 생기는 흔들림
            if group == "PD":
                values = values + pd_shift
            values[0] += 0.06 * (age - 65)                   # stride_time_cv는 나이와도 관련
            rows.append({"participant_id": f"P{i + 1:03d}", "group": group,
                         "age": round(age, 1), "sex": sex, "repeat": rep,
                         **dict(zip(FEATURES, values.round(3)))})
    return pd.DataFrame(rows)

df = make_demo_table()

# 🔁 실제 데이터로 바꿀 지점 ---------------------------------------------
# 4주차(Day 26)에 만든 feature table이 있으면 위의 df = make_demo_table() 대신:
# df = pd.read_csv("data/physionet_pd/derived/pd_features_records.csv")   # 내 파일 경로로
# FEATURES = ["내_feature_1", "내_feature_2"]                # 내 feature 열 이름으로
# 그리고 아래 두 이름을 내 표의 열 이름에 맞춘다.
ID_COL = "participant_id"   # 참가자 ID 열
GROUP_COL = "group"         # 진단군 열 (이 예시에서는 "PD" / "HC")
# -------------------------------------------------------------------------

from sklearn.model_selection import GroupShuffleSplit

X = df[FEATURES]                              # 모델에게 보여 줄 정보
y = (df[GROUP_COL] == "PD").astype(int)       # 정답: PD = 1, HC = 0   🔁 내 군 이름으로
groups = df[ID_COL]                           # 각 행의 참가자 (나눌 때만 사용)

# 참가자의 30%를 평가용으로 떼어 둔다 (같은 사람은 한쪽에만 들어간다)
splitter = GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=2)
train_idx, test_idx = next(splitter.split(X, y, groups=groups))

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

train_people = set(groups.iloc[train_idx])
test_people = set(groups.iloc[test_idx])
print("학습용: 참가자", len(train_people), "명 /", len(train_idx), "행")
print("평가용: 참가자", len(test_people), "명 /", len(test_idx), "행")
print("양쪽에 모두 있는 참가자 수:", len(train_people & test_people))
print("평가용 중 PD 비율:", round(y_test.mean(), 2))

assert len(train_people & test_people) == 0, "같은 참가자가 학습용과 평가용에 모두 있습니다!"

### 예시 2 — 로지스틱 회귀 baseline과 더미 모델

예시 1에 이어서 실행합니다.

In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

# 위약군: 학습 데이터에서 더 많은 군만 답하는 모델
dummy = DummyClassifier(strategy="most_frequent").fit(X_train, y_train)

# baseline: 표준화 → 로지스틱 회귀
baseline = make_pipeline(StandardScaler(), LogisticRegression())
baseline.fit(X_train, y_train)                 # 학습: 정답이 붙은 예제로 규칙 배우기

print(f"더미 모델      평가용 accuracy = {dummy.score(X_test, y_test):.2f}")
print(f"로지스틱 회귀  학습용 accuracy = {baseline.score(X_train, y_train):.2f}")
print(f"로지스틱 회귀  평가용 accuracy = {baseline.score(X_test, y_test):.2f}")

# 처음 보는 참가자 몇 명의 예측 확률 들여다보기 (측정 6개의 평균)
proba = baseline.predict_proba(X_test)[:, 1]   # 각 측정이 PD군(1)일 확률
peek = pd.DataFrame({"participant": groups.iloc[test_idx].values,
                     "정답": y_test.values, "PD군일_확률": proba})
print()
print(peek.groupby("participant").mean().round(2).head(6))

# 계수: 이 모델이 각 feature에 준 가중치 (표준화된 단위)
coefs = pd.Series(baseline[-1].coef_[0], index=FEATURES).sort_values(key=abs, ascending=False)
print()
print("계수 (양수 = 값이 클수록 PD군 쪽으로):")
print(coefs.round(2))

### 예시 3 — (선택) 랜덤 포레스트로 같은 일을 해 보기

실라부스는 로지스틱 회귀와 랜덤 포레스트 **중 하나**를 고르라고 합니다. 이 코스는 로지스틱 회귀를 주 baseline으로 쓰지만, scikit-learn에서는 모델을 바꿔 끼우는 것이 한 줄이라는 것을 보여 주기 위해 한 번 돌려 봅니다. 예시 2에 이어서 실행합니다.

In [ ]:
from sklearn.ensemble import RandomForestClassifier

forest = RandomForestClassifier(n_estimators=200, random_state=0)
forest.fit(X_train, y_train)

print(f"랜덤 포레스트  학습용 accuracy = {forest.score(X_train, y_train):.2f}")
print(f"랜덤 포레스트  평가용 accuracy = {forest.score(X_test, y_test):.2f}")
print()
print("feature 중요도:")
print(pd.Series(forest.feature_importances_, index=FEATURES).sort_values(ascending=False).round(2))